In [1]:
"""
Step 15 — Pool + cache raw eGeMAPS for ALL 275 participants
(official train/dev/test) from edaic_btp_dataset.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ------------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------------
DATASET_ROOT = Path(
    "/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset"
)

print("Using DATASET_ROOT:", DATASET_ROOT)

RAW_DIR = DATASET_ROOT / "raw_egemaps"
LABELS_DIR = DATASET_ROOT / "labels"

# Cache is stored in Kaggle working directory
CACHE_DIR = Path("/kaggle/working/pooled_egemaps_full")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

POOL_FACTOR = 250  # average every 250 consecutive frames


def egemaps_path(pid):
    matches = list((RAW_DIR / f"{pid}_P").glob("*egemaps.csv"))
    return matches[0] if matches else None


def pool_frames(arr, factor):
    n_full = (arr.shape[0] // factor) * factor

    if n_full == 0:
        return arr.mean(axis=0, keepdims=True)

    return arr[:n_full].reshape(
        -1, factor, arr.shape[1]
    ).mean(axis=1)


# ------------------------------------------------------------------
# Load all three official splits
# ------------------------------------------------------------------
all_labels = []

for split_name in ["train", "dev", "test"]:
    d = pd.read_csv(LABELS_DIR / f"{split_name}_split.csv")

    d["Split"] = split_name

    all_labels.append(
        d[["Participant_ID", "PHQ_Binary", "Split"]]
    )

labels_df = pd.concat(all_labels, ignore_index=True)

print(f"Total labeled participants: {len(labels_df)}")

print(
    labels_df.groupby("Split")["PHQ_Binary"].value_counts()
)


# ------------------------------------------------------------------
# Pool + cache each participant
# ------------------------------------------------------------------
feature_cols = None
failed = []

for pid in tqdm(
    labels_df["Participant_ID"].tolist(),
    desc="Pooling eGeMAPS"
):

    out_path = CACHE_DIR / f"{pid}.npy"

    # Skip participants that have already been processed
    if out_path.exists():
        continue

    p = egemaps_path(pid)

    if p is None:
        failed.append((pid, "raw file not found"))
        continue

    try:
        d = pd.read_csv(p, sep=";")

        # Detect feature columns only once
        if feature_cols is None:
            feature_cols = [
                c for c in d.columns
                if c not in ("name", "frameTime")
            ]

            print("\nFeature columns:")
            print(feature_cols)
            print("Number of features:", len(feature_cols))

        arr = d[feature_cols].values.astype(np.float32)

        pooled = pool_frames(
            arr,
            POOL_FACTOR
        )

        np.save(out_path, pooled)

    except Exception as e:
        failed.append((pid, str(e)))


# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------
cached = len(list(CACHE_DIR.glob("*.npy")))

print(f"\nCached: {cached} / {len(labels_df)}")

if failed:
    print(f"Failed ({len(failed)}):")
    print(failed[:10])

Using DATASET_ROOT: /kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset
Total labeled participants: 275
Split  PHQ_Binary
dev    0              44
       1              12
test   0              39
       1              17
train  0             126
       1              37
Name: count, dtype: int64


Pooling eGeMAPS:   0%|          | 1/275 [00:00<02:35,  1.76it/s]


Feature columns:
['Loudness_sma3', 'alphaRatio_sma3', 'hammarbergIndex_sma3', 'slope0-500_sma3', 'slope500-1500_sma3', 'spectralFlux_sma3', 'mfcc1_sma3', 'mfcc2_sma3', 'mfcc3_sma3', 'mfcc4_sma3', 'F0semitoneFrom27.5Hz_sma3nz', 'jitterLocal_sma3nz', 'shimmerLocaldB_sma3nz', 'HNRdBACF_sma3nz', 'logRelF0-H1-H2_sma3nz', 'logRelF0-H1-A3_sma3nz', 'F1frequency_sma3nz', 'F1bandwidth_sma3nz', 'F1amplitudeLogRelF0_sma3nz', 'F2frequency_sma3nz', 'F2amplitudeLogRelF0_sma3nz', 'F3frequency_sma3nz', 'F3amplitudeLogRelF0_sma3nz']
Number of features: 23


Pooling eGeMAPS: 100%|██████████| 275/275 [02:37<00:00,  1.74it/s]


Cached: 275 / 275


In [2]:
"""
Step 16 — BiLSTM on the OFFICIAL train/dev/test split (163/56/56),
directly comparable to the classical F1=0.632 baseline.
"""

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
CACHE_DIR = Path("/kaggle/working/pooled_egemaps_full")

DATASET_ROOT = Path(
    "/kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset"
)

LABELS_DIR = DATASET_ROOT / "labels"

print("Using DATASET_ROOT:", DATASET_ROOT)
print("Using LABELS_DIR:", LABELS_DIR)


# ------------------------------------------------------------------
# 1. Load official splits, filter to cached participants
# ------------------------------------------------------------------
def load_split(name):
    d = pd.read_csv(
        LABELS_DIR / f"{name}_split.csv"
    )[["Participant_ID", "PHQ_Binary"]]

    d["path"] = d["Participant_ID"].apply(
        lambda pid: CACHE_DIR / f"{pid}.npy"
    )

    # Keep only participants whose pooled features were successfully cached
    d = d[
        d["path"].apply(lambda p: p.exists())
    ].reset_index(drop=True)

    return d


train_df = load_split("train")
dev_df = load_split("dev")
test_df = load_split("test")

print(
    f"train:{len(train_df)} "
    f"dev:{len(dev_df)} "
    f"test:{len(test_df)}"
)

for name, d in [
    ("train", train_df),
    ("dev", dev_df),
    ("test", test_df)
]:
    print(
        f"  {name}:",
        d.PHQ_Binary.value_counts().to_dict()
    )


# ------------------------------------------------------------------
# 2. Normalization statistics from TRAIN only
# ------------------------------------------------------------------
all_train_frames = np.concatenate(
    [np.load(p) for p in train_df["path"]],
    axis=0
)

frame_mean = all_train_frames.mean(axis=0)
frame_std = all_train_frames.std(axis=0) + 1e-6

n_feats = frame_mean.shape[0]

print(f"Feature dim: {n_feats}")


# ------------------------------------------------------------------
# 3. Dataset + collate
# ------------------------------------------------------------------
class PooledSeqDataset(Dataset):

    def __init__(self, df):
        self.paths = df["path"].tolist()
        self.labels = df["PHQ_Binary"].values.astype(np.float32)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        arr = (
            np.load(self.paths[idx])
            - frame_mean
        ) / frame_std

        return (
            torch.tensor(arr, dtype=torch.float32),
            self.labels[idx]
        )


def collate_fn(batch):

    seqs, labels = zip(*batch)

    lengths = torch.tensor(
        [len(s) for s in seqs]
    )

    padded_seqs = pad_sequence(
        seqs,
        batch_first=True
    )

    labels = torch.tensor(
        labels,
        dtype=torch.float32
    )

    return padded_seqs, lengths, labels


BATCH_SIZE = 8

train_loader = DataLoader(
    PooledSeqDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    PooledSeqDataset(dev_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    PooledSeqDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


# ------------------------------------------------------------------
# 4. Model
#    Small BiLSTM + attention
# ------------------------------------------------------------------
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=32,
        dropout=0.4
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Linear(
            hidden_dim * 2,
            1
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            1
        )

    def forward(self, x, lengths):

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        out, _ = self.lstm(packed)

        out, _ = pad_packed_sequence(
            out,
            batch_first=True
        )

        # Mask padded positions
        mask = (
            torch.arange(
                out.size(1),
                device=out.device
            )[None, :]
            <
            lengths[:, None].to(out.device)
        )

        # Attention scores
        attn_scores = (
            self.attn(out)
            .squeeze(-1)
            .masked_fill(
                ~mask,
                float("-inf")
            )
        )

        # Attention weights
        attn_weights = (
            torch.softmax(
                attn_scores,
                dim=1
            )
            .unsqueeze(-1)
        )

        # Weighted sequence representation
        pooled = (
            out * attn_weights
        ).sum(dim=1)

        # Binary classification
        return self.classifier(
            self.dropout(pooled)
        ).squeeze(-1)


model = BiLSTMClassifier(
    input_dim=n_feats
).to(DEVICE)


# ------------------------------------------------------------------
# 5. Class-weighted loss
# ------------------------------------------------------------------
n_pos = train_df.PHQ_Binary.sum()
n_neg = len(train_df) - n_pos

pos_weight = torch.tensor(
    [n_neg / n_pos],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


# ------------------------------------------------------------------
# 6. Evaluation function
# ------------------------------------------------------------------
def evaluate(loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, lengths, y in loader:

            x = x.to(DEVICE)
            lengths = lengths.to(DEVICE)

            logits = model(
                x,
                lengths
            )

            preds = (
                torch.sigmoid(logits) >= 0.5
            ).float().cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y.numpy())

    return {
        "Accuracy": accuracy_score(
            all_labels,
            all_preds
        ),

        "Precision": precision_score(
            all_labels,
            all_preds,
            zero_division=0
        ),

        "Recall": recall_score(
            all_labels,
            all_preds,
            zero_division=0
        ),

        "F1": f1_score(
            all_labels,
            all_preds,
            zero_division=0
        )
    }


# ------------------------------------------------------------------
# 7. Train with early stopping on DEV F1
# ------------------------------------------------------------------
best_dev_f1 = -1
best_state = None

patience = 10
patience_ctr = 0

for epoch in range(80):

    model.train()

    total_loss = 0

    for x, lengths, y in train_loader:

        x = x.to(DEVICE)
        lengths = lengths.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(
            x,
            lengths
        )

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    # Evaluate on dev set
    dev_metrics = evaluate(dev_loader)

    avg_loss = (
        total_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch + 1:2d} | "
        f"loss={avg_loss:.4f} | "
        f"dev_F1={dev_metrics['F1']:.4f} "
        f"dev_recall={dev_metrics['Recall']:.4f}"
    )

    # Save best model based on DEV F1
    if dev_metrics["F1"] > best_dev_f1:

        best_dev_f1 = dev_metrics["F1"]

        best_state = {
            k: v.clone()
            for k, v in model.state_dict().items()
        }

        patience_ctr = 0

    else:

        patience_ctr += 1

        if patience_ctr >= patience:

            print(
                f"Early stopping at epoch {epoch + 1} "
                f"(best dev F1={best_dev_f1:.4f})"
            )

            break


# ------------------------------------------------------------------
# 8. Load best model and report final results
# ------------------------------------------------------------------
model.load_state_dict(best_state)

print(
    "\n================ BiLSTM "
    "(official split) RESULTS ================"
)

dev_results = evaluate(dev_loader)
test_results = evaluate(test_loader)

print(
    "Dev :",
    {
        k: round(v, 4)
        for k, v in dev_results.items()
    }
)

print(
    "Test:",
    {
        k: round(v, 4)
        for k, v in test_results.items()
    }
)

Using device: cpu
Using DATASET_ROOT: /kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset
Using LABELS_DIR: /kaggle/input/datasets/am2008/edaic-full-data/edaic_btp_dataset/labels
train:163 dev:56 test:56
  train: {0: 126, 1: 37}
  dev: {0: 44, 1: 12}
  test: {0: 39, 1: 17}
Feature dim: 23
Epoch  1 | loss=1.0666 | dev_F1=0.4348 dev_recall=0.4167
Epoch  2 | loss=1.0701 | dev_F1=0.4000 dev_recall=0.5000
Epoch  3 | loss=1.0876 | dev_F1=0.4118 dev_recall=0.5833
Epoch  4 | loss=1.0440 | dev_F1=0.4324 dev_recall=0.6667
Epoch  5 | loss=1.0431 | dev_F1=0.4103 dev_recall=0.6667
Epoch  6 | loss=1.0257 | dev_F1=0.4000 dev_recall=0.6667
Epoch  7 | loss=1.0110 | dev_F1=0.4000 dev_recall=0.6667
Epoch  8 | loss=0.9859 | dev_F1=0.4211 dev_recall=0.6667
Epoch  9 | loss=0.9909 | dev_F1=0.4324 dev_recall=0.6667
Epoch 10 | loss=0.9464 | dev_F1=0.4324 dev_recall=0.6667
Epoch 11 | loss=0.9186 | dev_F1=0.3784 dev_recall=0.5833
Early stopping at epoch 11 (best dev F1=0.4348)

================ BiLST